# Silver — Exchange Rates & OMX

Cleans Bronze market data: drops null rows, derives ISK/EUR cross-rate, and writes a production-ready table.

**Source:** `bronze.yahoo_finance_raw`  
**Output:** `silver.exchange_rates` (Delta table)  
**Key derivation:** `close_iskeur = close_iskusd_x / close_eurusd_x`  

In [ ]:
from pyspark.sql import functions as F

df = spark.read.format("delta").table("bronze.yahoo_finance_raw")

print(f"Bronze rows: {df.count()}")
df.printSchema()

In [ ]:
# Drop rows missing core FX fields
df_clean = df.dropna(subset=["close_eurusd_x", "close_iskusd_x"])

# Derive ISK/EUR cross-rate (ISKEUR=X ticker returns all NaN from Yahoo Finance)
df_clean = df_clean.withColumn(
    "close_iskeur",
    F.round(F.col("close_iskusd_x") / F.col("close_eurusd_x"), 6)
)

df_clean = df_clean.withColumn("processed_at", F.current_timestamp())

dropped = df.count() - df_clean.count()
print(f"Rows in: {df.count()} | Rows out: {df_clean.count()} | Dropped: {dropped}")
df_clean.select("date", "close_eurusd_x", "close_iskusd_x", "close_iskeur", "close__omx").show(5)

In [ ]:
df_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.exchange_rates")

print(f"Saved to silver.exchange_rates — {df_clean.count()} rows")